# CSE 151B Competition — Fine-Tuning Notebook

This notebook implements **QLoRA supervised fine-tuning (SFT)** on Qwen3-4B-Thinking-2507 to improve math reasoning accuracy beyond the 74% prompt-engineering baseline.

**Pipeline:**
1. Environment setup (`peft`, `trl`, `datasets`)
2. Configuration
3. Load base model (4-bit QLoRA via BitsAndBytes — NOT vLLM)
4. Apply LoRA adapters with PEFT
5. Curate training data (competition correct traces + external math datasets)
6. SFT training with `SFTTrainer`
7. Merge adapter and quick sanity eval on the 100-item eval subset
8. Instructions for using the fine-tuned model in the main inference notebook

> **After training**, point `MODEL_ID` in `starter_code_cse151b_comp.ipynb` to `./checkpoints/qwen3-lora-merged` — all downstream prompt routing, two-stage MCQ flow, and scoring code stays unchanged.

## 1. Environment Setup

Install fine-tuning dependencies on top of the existing `cse151b` kernel.

> **Run this cell once**, then restart the kernel.

In [ ]:
# # Run once — comment out after first installation
# !.venv/bin/python -m pip install \
#     peft>=0.11.0 \
#     trl>=0.9.0 \
#     datasets>=2.19.0 \
#     accelerate>=0.30.0 \
#     bitsandbytes>=0.43.0
# print("Done. Restart kernel before proceeding.")

In [ ]:
# Activate venv — run every time
!source ./.venv/bin/activate

## 2. Configuration

All hyperparameters in one place. Tune here before running training.

In [ ]:
import os
import json
import re
import sys
import hashlib
from pathlib import Path
from typing import Optional

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen3-4B-Thinking-2507"
ADAPTER_OUTPUT = "checkpoints/qwen3-lora"        # LoRA adapter saved here
MERGED_OUTPUT  = "checkpoints/qwen3-lora-merged"  # merged weights for vLLM
GPU_ID         = "0"

# ── Data ──────────────────────────────────────────────────────────────────────
DATA_PATH      = "data/public.jsonl"              # competition questions
RESULTS_PATH   = "results/starter_results.jsonl"  # filter to correct=True
SEED           = 13

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Training hyperparameters ──────────────────────────────────────────────────
MAX_SEQ_LEN    = 4096   # shorter than inference; raise to 8192 if VRAM allows
BATCH_SIZE     = 2
GRAD_ACCUM     = 8      # effective batch = 16
LR             = 2e-4
EPOCHS         = 2
WARMUP_RATIO   = 0.05
VAL_SPLIT      = 0.05   # 5% held out for eval loss tracking

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
print(f"Config: model={MODEL_ID}, r={LORA_R}, lr={LR}, epochs={EPOCHS}, max_seq={MAX_SEQ_LEN}")

## 3. Load Base Model (4-bit QLoRA)

We use HuggingFace `transformers` + BitsAndBytes **4-bit NF4** quantization — NOT vLLM. Training requires gradient flow through the model; vLLM is inference-only.

| Precision | VRAM (Qwen3-4B) | Use case |
|-----------|----------------|----------|
| BF16      | ~8 GB           | Full fine-tune (needs high-memory GPU) |
| INT8      | ~4 GB           | vLLM inference (main notebook) |
| **NF4 4-bit** | **~3 GB** | **QLoRA training ← this notebook** |

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,  # nested quantization saves ~0.4 GB
    bnb_4bit_quant_type="nf4",       # NF4 > INT4 for normal-distributed weights
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for SFTTrainer causal LM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded on: {next(model.parameters()).device}")
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 4. Apply LoRA with PEFT

`prepare_model_for_kbit_training` casts layer norms to float32 and enables gradient checkpointing — both required for stable QLoRA training.

We target all attention + MLP projection layers (`q/k/v/o_proj`, `gate/up/down_proj`). This captures ~1.7% of total parameters (~67M of 3.9B), which is sufficient for task adaptation without catastrophic forgetting.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params: ~67M || all params: ~3.9B || trainable%: ~1.7%

## 5. Data Curation

We mix two sources:

| Source | Size | Role |
|--------|------|------|
| Competition `public.jsonl` correct traces | ~832 examples | Domain alignment (same question distribution as test set) |
| GSM8K + MATH (`hendrycks/competition_math`) | ~15K examples | Diverse math reasoning breadth |

**Why use the model's own correct responses as training data?**  
The model already knows how to reason — we just need to reinforce the traces that led to correct answers. This is rejection-sampling SFT (a lightweight form of RLHF).

**Format**: Every example is formatted as a chat with the same system prompt used during inference, so the fine-tuned model behaves identically under the main notebook's prompt routing.

In [ ]:
# ── Reuse preprocessing + prompt constants from main notebook ─────────────────
# Copied verbatim to avoid .ipynb import dependency.

LATEX_CMDS = [
    "int", "iint", "iiint", "oint", "sum", "prod", "lim",
    "infty", "partial",
    "frac", "dfrac", "tfrac", "sqrt", "binom",
    "sin", "cos", "tan", "cot", "sec", "csc",
    "arcsin", "arccos", "arctan",
    "sinh", "cosh", "tanh",
    "log", "ln", "exp",
    "alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta",
    "theta", "iota", "kappa", "lambda", "mu", "nu",
    "xi", "pi", "rho", "sigma", "tau", "phi", "chi", "psi", "omega",
    "Gamma", "Delta", "Theta", "Lambda", "Pi", "Sigma", "Phi", "Psi", "Omega",
    "pm", "mp", "times", "cdot", "leq", "geq", "neq",
    "approx", "equiv", "sim", "to", "in", "notin",
    "subset", "supset", "cup", "cap",
    "mathbb", "mathrm", "mathbf", "mathcal",
    "left", "right", "text",
]
_LATEX_MATH_CONTEXT = r"[{}_^()]"
_LATEX_PATTERNS = [
    (re.compile(rf"(?<!\\)\b{cmd}(?={_LATEX_MATH_CONTEXT})"), rf"\\{cmd}")
    for cmd in LATEX_CMDS
]

def repair_latex(text: str) -> str:
    for pattern, repl in _LATEX_PATTERNS:
        text = pattern.sub(repl, text)
    return text

SYSTEM_PROMPT_MATH = (
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a "
    "single \\boxed{}, e.g. \\boxed{3, 7}."
)
SYSTEM_PROMPT_MULTIPART = (
    "This problem contains multiple sub-answers marked by [ANS] placeholders "
    "in the question. Identify each sub-question, then solve them in the order "
    "they appear.\n\n"
    "After completing all reasoning, output every answer in a SINGLE \\boxed{}, "
    "comma-separated, in the same order as the [ANS] slots — for example "
    "\\boxed{a, b, c}. Do not split answers across multiple \\boxed{} expressions."
)
SYSTEM_PROMPT_LONG = (
    "Before solving, restate the problem in your own words and list every given "
    "condition and the quantity to find. Then solve.\n\n"
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a "
    "single \\boxed{}, e.g. \\boxed{3, 7}."
)

def pick_system_prompt(question: str) -> str:
    """Mirror the free-form routing logic from select_prompt() in the main notebook."""
    if question.count("[ANS]") > 1:
        return SYSTEM_PROMPT_MULTIPART
    if len(question) > 500:
        return SYSTEM_PROMPT_LONG
    return SYSTEM_PROMPT_MATH

print("Preprocessing utilities loaded.")

In [ ]:
# ── Source 1: Competition correct traces ──────────────────────────────────────
comp_questions = {d["id"]: d for d in (json.loads(l) for l in open(DATA_PATH))}

comp_examples = []
if Path(RESULTS_PATH).exists():
    for line in open(RESULTS_PATH):
        r = json.loads(line)
        if not r.get("correct"):
            continue
        qid = r["id"]
        if qid not in comp_questions:
            continue
        item = comp_questions[qid]
        # Skip MCQ: response is just \boxed{LETTER}, not a useful reasoning trace
        if item.get("options"):
            continue
        question = repair_latex(item["question"])
        system   = pick_system_prompt(question)
        comp_examples.append({
            "text": tokenizer.apply_chat_template(
                [{"role": "system",    "content": system},
                 {"role": "user",      "content": question},
                 {"role": "assistant", "content": r["response"]}],
                tokenize=False,
            )
        })
else:
    print(f"WARNING: {RESULTS_PATH} not found. Run the main notebook to generate it.")
    print("Proceeding with external datasets only.")

print(f"Competition correct traces (free-form only): {len(comp_examples)}")

In [ ]:
# ── Source 2: External math datasets ─────────────────────────────────────────
from datasets import load_dataset

ext_examples = []

# GSM8K: grade-school math word problems with step-by-step solutions
gsm8k = load_dataset("gsm8k", "main", split="train")
for row in gsm8k:
    q = repair_latex(row["question"])
    # GSM8K answers end in "#### <number>" — convert to \boxed{} to match model style
    parts  = row["answer"].split("####")
    chain  = parts[0].strip()
    number = parts[-1].strip().replace(",", "")
    a = chain + f"\n\nTherefore, the answer is $\\boxed{{{number}}}$."
    ext_examples.append({
        "text": tokenizer.apply_chat_template(
            [{"role": "system",    "content": SYSTEM_PROMPT_MATH},
             {"role": "user",      "content": q},
             {"role": "assistant", "content": a}],
            tokenize=False,
        )
    })

print(f"GSM8K: {len(ext_examples)} examples")

# MATH (Hendrycks competition math): harder olympiad-style with full \boxed{} solutions
math_ds = load_dataset("hendrycks/competition_math", split="train", trust_remote_code=True)
math_start = len(ext_examples)
for row in math_ds:
    q = repair_latex(row["problem"])
    a = row["solution"]  # already contains full chain-of-thought + \boxed{}
    ext_examples.append({
        "text": tokenizer.apply_chat_template(
            [{"role": "system",    "content": pick_system_prompt(q)},
             {"role": "user",      "content": q},
             {"role": "assistant", "content": a}],
            tokenize=False,
        )
    })

print(f"MATH (competition_math): {len(ext_examples) - math_start} examples")
print(f"Total external: {len(ext_examples)}")

In [ ]:
# ── Deduplicate, mix, and split ───────────────────────────────────────────────
import random
from datasets import Dataset

rng = random.Random(SEED)

seen = set()
all_examples = []
for ex in comp_examples + ext_examples:
    h = hashlib.md5(ex["text"].encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        all_examples.append(ex)

rng.shuffle(all_examples)

n_val = max(1, int(len(all_examples) * VAL_SPLIT))
val_examples   = all_examples[:n_val]
train_examples = all_examples[n_val:]

train_ds = Dataset.from_list(train_examples)
val_ds   = Dataset.from_list(val_examples)

print(f"Total after dedup: {len(all_examples)}")
print(f"  train: {len(train_ds)}   val: {len(val_ds)}")

# Sanity-check one example
print("\n── Sample training example (first 400 chars) ──")
print(train_ds[0]["text"][:400])

## 6. SFT Training

Key decisions:
- **`bf16=True`** — numerically stable for Qwen3; `fp16` can cause NaN on long sequences
- **`cosine` LR schedule** — decays smoothly over 2 epochs; works well for small datasets
- **`packing=False`** — variable-length math responses; packing can corrupt CoT traces
- **`report_to="none"`** — no external logging dependency (wandb, etc.)

Expected training time on DSMLP `gpu-class=medium` (A100 40GB): ~1.5–3 hours for ~15K examples × 2 epochs.

In [ ]:
from trl import SFTTrainer, SFTConfig

Path(ADAPTER_OUTPUT).mkdir(parents=True, exist_ok=True)

training_args = SFTConfig(
    output_dir=ADAPTER_OUTPUT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

print("Starting training...")
trainer.train()

trainer.save_model(ADAPTER_OUTPUT)
tokenizer.save_pretrained(ADAPTER_OUTPUT)
print(f"\nLoRA adapter saved to: {ADAPTER_OUTPUT}")

## 7. Merge Adapter & Quick Eval

Merge the LoRA adapter into the base model weights, save as a standalone checkpoint, then run on the 100-item eval subset using `judger.py`.

**Why merge before eval?** Merged weights can be loaded by vLLM with `MODEL_ID = "./checkpoints/qwen3-lora-merged"`, keeping the main notebook's fast batched inference intact.

> If VRAM is tight after training, restart the kernel before this section.

In [ ]:
from peft import PeftModel

print("Loading base model in bf16 for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("Applying LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_OUTPUT)
merged_model = merged_model.merge_and_unload()  # fuse weights in-place

Path(MERGED_OUTPUT).mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(MERGED_OUTPUT)
tokenizer.save_pretrained(MERGED_OUTPUT)
print(f"Merged model saved to: {MERGED_OUTPUT}")

In [ ]:
# ── Quick sanity eval on the 100-item stratified subset ───────────────────────
# Uses the same judger.py scoring as the main notebook.
# This cell is optional — you can also run the main notebook with
# MODEL_ID = MERGED_OUTPUT to use its faster vLLM batched inference instead.

from transformers import pipeline
from tqdm import tqdm

EVAL_SUBSET_PATH = "results/eval_subset.json"

if not Path(EVAL_SUBSET_PATH).exists():
    print(f"Eval subset not found at {EVAL_SUBSET_PATH}. Run the main notebook first to generate it.")
else:
    with open(EVAL_SUBSET_PATH) as f:
        subset_ids = set(json.load(f))

    all_data  = [json.loads(l) for l in open(DATA_PATH)]
    eval_items = [d for d in all_data if d["id"] in subset_ids]

    gen_pipe = pipeline(
        "text-generation",
        model=merged_model,
        tokenizer=tokenizer,
        device_map="auto",
    )

    eval_responses = []
    for item in tqdm(eval_items, desc="Generating (fine-tuned)"):
        question = repair_latex(item["question"])
        system   = pick_system_prompt(question)
        prompt   = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": question}],
            tokenize=False,
            add_generation_prompt=True,
        )
        out = gen_pipe(
            prompt, max_new_tokens=2048, temperature=0.6, top_p=0.95,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
        eval_responses.append(out[0]["generated_text"][len(prompt):])

    # Score with judger.py (same logic as main notebook Section 7)
    sys.path.insert(0, ".")
    from judger import Judger
    judger = Judger(strict_extract=False)

    def extract_letter(text):
        m = re.search(r"\\boxed\{([A-Za-z])\}", text)
        if m:
            return m.group(1).upper()
        matches = re.findall(r"\b([A-Z])\b", text.upper())
        return matches[-1] if matches else ""

    results_ft = []
    for item, resp in zip(eval_items, eval_responses):
        is_mcq = bool(item.get("options"))
        gold   = item["answer"]
        if is_mcq:
            correct = extract_letter(resp) == str(gold).strip().upper()
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(pred=resp, gold=gold_list, options=[[]] * len(gold_list))
            except Exception:
                correct = False
        results_ft.append({"id": item["id"], "is_mcq": is_mcq, "correct": correct})

    mcq_r  = [r for r in results_ft if r["is_mcq"]]
    free_r = [r for r in results_ft if not r["is_mcq"]]
    acc    = lambda s: sum(r["correct"] for r in s) / len(s) * 100 if s else 0.0

    print("\n" + "=" * 52)
    print("FINE-TUNED MODEL EVALUATION RESULTS")
    print("=" * 52)
    print(f"  MCQ        : {sum(r['correct'] for r in mcq_r):3d} / {len(mcq_r):3d}  ({acc(mcq_r):.2f}%)")
    print(f"  Free-form  : {sum(r['correct'] for r in free_r):3d} / {len(free_r):3d}  ({acc(free_r):.2f}%)")
    print(f"  Overall    : {sum(r['correct'] for r in results_ft):3d} / {len(results_ft):3d}  ({acc(results_ft):.2f}%)")
    print("=" * 52)
    print(f"  Baseline (prompt engineering only): 74.00%")
    delta = acc(results_ft) - 74.0
    print(f"  Delta vs baseline: {delta:+.2f}pp")

## 8. Using the Fine-Tuned Model in the Main Notebook

The merged model is a drop-in replacement for the base model. **Only one line** changes in `starter_code_cse151b_comp.ipynb`:

```python
# Before (base model via HuggingFace Hub):
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# After (fine-tuned, local path):
MODEL_ID = "./checkpoints/qwen3-lora-merged"
```

Everything else — LaTeX repair, prompt routing (5 slots), two-stage MCQ flow, scoring — remains unchanged.

### Checkpoint inventory

| Path | Contents | Use |
|------|----------|-----|
| `checkpoints/qwen3-lora/` | LoRA adapter weights only (~67M params) | Resume training, share adapter |
| `checkpoints/qwen3-lora-merged/` | Full merged model (bf16, ~8GB) | vLLM inference in main notebook |

### DSMLP launch command (unchanged)
```bash
launch-sp26-cuda128.sh -l gpu-class=medium -W CSE151B_SP26_A00 -g 1 -c 8 -m 32
```

### Next steps after SFT
- **Self-consistency** — run the main notebook's self-consistency variant (`starter_code_cse151b_comp_self_consistency.ipynb`) with the merged model
- **GRPO** — use `judger.auto_judge` as a reward function for reinforcement learning (see PLAN.md Phase 4)